# MZ세대 공무원의 이직의도: RuleFit 기반 설명가능 인공지능

- 2024 공직생활실태조사로 MZ세대(1981년 이후 출생) 공무원의 이직의도 예측
- RuleFit으로 "어떤 조건의 공무원이 이직의도가 높은가"를 규칙 형태로 해석

> 신연우, 김진영, 이태헌 (2025). MZ세대 공무원의 이직의도 분석: RuleFit 기반 설명가능 인공지능(XAI)을 중심으로. *사회과학연구논총*, 41(2), 195-241.

- 필요한 파일: 2024 공직생활실태조사 원자료 (`.sav` 또는 `.xlsx`), 경로는 아래 `DATA_PATH`에 입력
- 분석 순서: 전처리 → 기술통계와 신뢰도 → 상관 → PyCaret 모형 비교와 랜덤 포레스트 튜닝 → RuleFit 규칙 추출 → 상위 규칙 시각화

## 1. 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, skew, kurtosis

DATA_PATH = "../data/gong2024.sav"   # 원자료 경로 (.sav 또는 .xlsx)

In [ ]:
# 한글 그래프 글꼴 (NanumGothic이 설치되어 있어야 함)
sns.set(font_scale=0.8, style="whitegrid")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
def load_raw(path):
    """원자료 읽기 (.sav면 변수 값 라벨도 함께 반환)"""
    if path.endswith(".sav"):
        import pyreadstat
        df, meta = pyreadstat.read_sav(path)
        return df, meta.variable_value_labels
    return pd.read_excel(path), None

df, value_labels = load_raw(DATA_PATH)
df.shape

## 2. 전처리

### 2.1. 변수 정의

- 여러 문항으로 측정한 요인은 문항 평균, 단일 문항 변수와 인구통계 변수는 원래 값 사용

In [ ]:
# 부정 방향으로 묻는 문항 (1~5점을 뒤집음)
REVERSE_ITEMS = ["q8_5", "q14_4", "q19_5", "q33_5", "q33_6", "q33_7", "q33_8"]

# 변수 이름: 원문항 (문항이 여러 개면 평균)
VARIABLES = {
    "업무량": ["q1"],
    "근무환경": ["q2"],
    "자원제공": ["q3_1", "q3_2", "q3_3"],
    "업무자율성": ["q4_1", "q4_2", "q4_3"],
    "채용": ["q7_1", "q7_2", "q7_3", "q7_4"],
    "승진": ["q8_1", "q8_2", "q8_3", "q8_4", "q8_5"],
    "배치전환": ["q10_1", "q10_2", "q10_3"],
    "보수 및 보상": ["q11_1", "q11_2", "q11_3", "q11_4"],
    "호봉제": ["q12"],
    "후생복지": ["q13_1", "q13_2"],
    "교육훈련/능력발전": ["q14_1", "q14_2", "q14_3", "q14_4"],
    "업무 수행 역량": ["q15_1", "q15_2", "q15_3"],
    "전문성 수준": ["q16_1", "q16_2"],
    "조직목표": ["q19_1", "q19_2", "q19_3", "q19_4", "q19_5"],
    "리더십": ["q20_1", "q20_2", "q20_3", "q20_4", "q20_5", "q20_6", "q20_7"],
    "과업지향 조직문화": ["q21_1", "q21_2"],
    "혁신지향 조직문화": ["q21_3", "q21_7"],
    "관계지향 조직문화": ["q21_4", "q21_8"],
    "위계질서 조직문화": ["q21_5", "q21_6"],
    "조직 냉소주의": ["q21_9", "q21_10"],
    "조직 내 정치": ["q21_11", "q21_12"],
    "의사결정": ["q22_1", "q22_2", "q22_3", "q22_4"],
    "변화관리": ["q23_1", "q23_2", "q23_3"],
    "협업/의사소통": ["q24_1", "q24_2", "q24_3"],
    "조직성과": ["q25_1", "q25_2"],
    "성과관리": ["q26_1", "q26_2", "q26_3", "q26_4", "q26_5"],
    "업무성과": ["q27_1", "q27_2", "q27_3"],
    "부정부패": ["q29"],
    "공공봉사동기": ["q30_1", "q30_2", "q30_3", "q30_4", "q30_5", "q30_6"],
    "역할갈등": ["q31_1", "q31_2", "q31_3"],
    "역할모호성": ["q31_4", "q31_5", "q31_6"],
    "업무 과중": ["q31_7"],
    "조직시민행동": ["q32_1", "q32_2", "q32_3", "q32_4", "q32_5"],
    "혁신행동": ["q33_1", "q33_2", "q33_3", "q33_4", "q33_5", "q33_6", "q33_7", "q33_8"],
    "공직가치": ["q34_1", "q34_2", "q34_3", "q34_4", "q34_5", "q34_6", "q34_7", "q34_8"],
    "조직몰입": ["q35_1", "q35_2", "q35_3", "q35_4"],
    "직무만족": ["q36_1", "q36_2", "q36_3"],
    "공직만족도": ["q37_1", "q37_2", "q37_3", "q37_4"],
    "삶의 질": ["q39_1", "q39_2"],
    # 인구통계, 근무 특성
    "성별": ["DM2"],
    "출생 연도": ["dq2"],
    "혼인상태": ["dq3"],
    "전체 자녀의 수": ["dq4_1"],
    "만 20세 미만 자녀의 수": ["dq4_2"],
    "입직당시 학력": ["dq5_1"],
    "현재 학력": ["dq5_2"],
    "입직 준비기간": ["dq6"],
    "재직기간": ["dq7"],
    "입직 시 채용 방식": ["dq8"],
    "입직당시 직급": ["dq9_1"],
    "현재 직급": ["dq9_2"],
    "1개월 평균 급여수준": ["dq10"],
    "1주일 평균 근무시간": ["dq11"],
    "기관유형": ["orgtype"],
    "기관구분": ["DM1"],
}

# 기술통계에서 빈도표로 볼 범주형 변수
CATEGORICAL = ["성별", "혼인상태", "입직당시 학력", "현재 학력", "입직 준비기간", "재직기간",
               "입직 시 채용 방식", "입직당시 직급", "현재 직급", "1개월 평균 급여수준",
               "1주일 평균 근무시간", "기관유형", "기관구분", "이직의도"]

### 2.2. 분석 자료 만들기

In [ ]:
def preprocess(df):
    # 1) MZ세대(1981년 이후 출생)만 선택
    mz = df[df["dq2"] >= 1981].copy()
    print("전체:", df.shape, "→ MZ세대:", mz.shape)

    # 2) 역문항 처리
    for col in REVERSE_ITEMS:
        mz[col] = 6 - mz[col]

    # 3) 요인 점수 계산 (문항 평균)
    data = pd.DataFrame({name: mz[items].mean(axis=1) for name, items in VARIABLES.items()},
                        index=mz.index)

    # 4) 이직의도 이진화: q38 1~3점 → 0(없음), 4~5점 → 1(있음)
    data["이직의도"] = np.select([mz["q38"] <= 3, mz["q38"] > 3], [0, 1], default=np.nan)
    print("분석 자료:", data.shape)
    return data

data = preprocess(df)
data["이직의도"].value_counts()

## 3. 기술통계와 신뢰도

In [ ]:
def cronbach_alpha(items):
    items = items.dropna()
    k = items.shape[1]
    if k <= 1:
        return np.nan
    return k / (k - 1) * (1 - items.var(ddof=1).sum() / items.sum(axis=1).var(ddof=1))

# 요인별 신뢰도 (역문항 처리 후, 전체 응답자 기준)
all_items = df.copy()
for col in REVERSE_ITEMS:
    all_items[col] = 6 - all_items[col]

alpha_table = pd.DataFrame(
    [{"변수": name, "문항 수": len(items), "Alpha": round(cronbach_alpha(all_items[items]), 3)}
     for name, items in VARIABLES.items() if len(items) >= 2]
).sort_values("변수").reset_index(drop=True)
alpha_table

In [ ]:
# 연속형 변수 기술통계 (첨도는 정규분포 = 3 기준)
numeric_data = data.drop(columns=CATEGORICAL)
summary = pd.DataFrame({
    "min": numeric_data.min(),
    "max": numeric_data.max(),
    "mode": numeric_data.apply(lambda s: s.mode().iloc[0]),
    "mean": numeric_data.mean(),
    "std": numeric_data.std(),
    "skewness": numeric_data.apply(skew),
    "kurtosis": numeric_data.apply(lambda s: kurtosis(s, fisher=True) + 3),
}).round(3).sort_index()
summary.to_csv("summary_stats.csv", encoding="utf-8-sig")
summary

In [ ]:
# 범주형 변수 빈도표
for col in CATEGORICAL:
    freq = data[col].value_counts(dropna=False)
    print(f"\n▶ {col}")
    print(pd.DataFrame({"빈도": freq, "비율(%)": (freq / len(data) * 100).round(2)}))

## 4. 상관

In [ ]:
numeric = data.select_dtypes(include="number")
corr = numeric.corr()

# 유의수준 표시: *** p ≤ .001, ** p ≤ .01, * p ≤ .05
stars = pd.DataFrame("", index=corr.index, columns=corr.columns)
for a in corr.index:
    for b in corr.columns:
        p = pearsonr(numeric[a], numeric[b])[1]
        stars.loc[a, b] = "***" if p <= .001 else "**" if p <= .01 else "*" if p <= .05 else ""

plt.figure(figsize=(20, 18))
sns.heatmap(corr, annot=stars, mask=np.triu(np.ones_like(corr, dtype=bool)), fmt="",
            cmap="coolwarm", center=0, linewidths=0.5, annot_kws={"fontweight": "bold"})
plt.xticks(fontsize=10, fontweight="bold")
plt.yticks(fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300)
plt.show()

## 5. 예측모형: PyCaret

- 학습 70%, 평가 30% 분할(PyCaret 기본값)
- 5겹 교차검증으로 여러 분류 모형을 AUC 기준으로 비교한 뒤 랜덤 포레스트 튜닝

In [ ]:
from pycaret.classification import setup, compare_models, create_model, tune_model, get_config

clf = setup(data=data, target="이직의도", session_id=42, fold=5, use_gpu=True)
best_model = compare_models(sort="AUC", n_select=1, fold=5)

In [ ]:
rf_model = create_model("rf")
tuned_rf = tune_model(rf_model, optimize="AUC", choose_better=True, n_iter=10)
tuned_rf

In [ ]:
X_train, y_train = get_config("X_train"), get_config("y_train")
X_test,  y_test  = get_config("X_test"),  get_config("y_test")

## 6. RuleFit

- 튜닝한 랜덤 포레스트와 같은 하이퍼파라미터로 트리 생성
- 트리의 경로를 "조건 & 조건" 형태의 규칙으로 변환한 뒤 L1 정규화 로지스틱 회귀로 중요한 규칙만 선택

- 규칙 중요도 = |계수| × √(지지도 × (1 − 지지도))
- 계수만 크거나 지지도(규칙이 적용되는 사례 비율)만 큰 규칙이 과대평가되지 않도록 둘을 함께 반영

In [ ]:
from rulefit import RuleFit
from sklearn.ensemble import RandomForestClassifier

# tune_model 결과의 하이퍼파라미터
rf = RandomForestClassifier(n_estimators=120, max_depth=9, max_features="sqrt",
                            min_samples_leaf=6, min_samples_split=5,
                            criterion="gini", bootstrap=True, n_jobs=-1, random_state=42)

rulefit = RuleFit(tree_generator=rf, rfmode="classification", random_state=19990423)
rulefit.fit(X_train.values, y_train.values, feature_names=X_train.columns)

In [ ]:
rules = rulefit.get_rules()
rules_nonzero = (rules[rules["importance"] != 0]
                 .sort_values("importance", ascending=False)
                 .round(3))
rules_nonzero.to_csv("rules_not_zero.csv", encoding="utf-8-sig")

print("0이 아닌 규칙 수:", len(rules_nonzero))
print("상위 5개 규칙 중요도 합:", rules_nonzero["importance"].head(5).sum())
print("전체 중요도 합:", rules_nonzero["importance"].sum())
rules_nonzero.head(10)

## 7. 상위 규칙 시각화

- 상위 규칙의 두 조건 변수별 이직의도 집단 분포를 바이올린 플롯으로 시각화
- 점선: 규칙의 분기 기준
- x축은 범주형 축이라 점선 위치는 기준값이 아니라 눈금 순서(0부터)로 지정

In [ ]:
def rule_violin(df, x, y, xline, yline, ylabel=None):
    plt.figure(figsize=(12, 7))
    sns.violinplot(x=x, y=y, hue="이직의도", data=df, split=True, inner="quartile",
                   palette={0: "#aecbfa", 1: "#f3b5a3"})
    plt.axvline(x=xline, color="gray", linestyle="--", linewidth=1.5)
    plt.axhline(y=yline, color="gray", linestyle="--", linewidth=1.5)
    plt.xlabel(x, fontsize=14, fontweight="bold")
    plt.ylabel(ylabel or y, fontsize=14, fontweight="bold")
    plt.xticks(fontsize=12, fontweight="bold")
    plt.yticks(fontsize=12, fontweight="bold")
    plt.grid(True, alpha=0.3, linewidth=0.5)
    plt.tight_layout()
    plt.show()

# 규칙 1: 전체 자녀의 수 <= 0.5 & 공직만족도 <= 2.625
rule_violin(data, x="공직만족도", y="전체 자녀의 수", xline=6.5, yline=0.5)

# 규칙 2: 공직만족도 > 2.625 & 직무만족 > 2.83 & 호봉제 <= 4.5 → 호봉제 조건을 만족하는 사례만
rule_violin(data[data["호봉제"] <= 4.5], x="공직만족도", y="직무만족",
            xline=6.5, yline=2.8, ylabel="직무만족도")

# 규칙 3: 조직 냉소주의 <= 3.25 & 업무 과중 <= 3.5
rule_violin(data, x="조직 냉소주의", y="업무 과중", xline=4.5, yline=3.5)